<a href="https://colab.research.google.com/github/grinaldo-oliveira/IC009/blob/main/Curadoria_de_Imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 🧪 Curadoria de Imagens

Neste notebook será feita a seleção de figuras e execução do augmentation de imagens.

Como a dimensão das imagens de teste é 112x122, as imagens para treino serão selecionadas a partir de uma dimensão mínima:

Pessoa → ≥ 48 px

Carro → ≥ 56 px

Moto → ≥ 72 px

Bicicleta → ≥ 80 px

---

## 🧪 Curadoria de Imagens para car, carSide e CarRear

Repositório https://universe.roboflow.com/side-recognizer/carside

Arquivo CarSide.v13i.yolov8-obb.
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/CarSide.v13i.yolov8-obb.zip' '/content/CarSide.v13i.yolov8-obb.zip'

In [ ]:
!unzip "/content/CarSide.v13i.yolov8-obb.zip"  -d /content/carside/

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/carside/train/labels/Auto663-D_NQ_NP_947219-MLA71009895185_082023-O-webp_jpg.rf.eb517d9273f7281d9e9aa5e8dd2488b8.txt  
  inflating: /content/carside/train/labels/Auto665-D_NQ_NP_714840-MLA70925987662_082023-O-webp_jpg.rf.fb1ab8b2c185e9fb45139d1fe6e0ab6a.txt  
  inflating: /content/carside/train/labels/Auto665-D_NQ_NP_818754-MLA70960560683_082023-O-webp_jpg.rf.d1ab77912451e565056f491bda0a226c.txt  
  inflating: /content/carside/train/labels/Auto665-D_NQ_NP_855473-MLA70925928732_082023-O-webp_jpg.rf.bd20cece9567ea01c7330cd205643aba.txt  
  inflating: /content/carside/train/labels/Auto665-D_NQ_NP_925709-MLA70925967966_082023-O-webp_jpg.rf.5227d2cc09b3cf7daa9bd6df7818732d.txt  
  inflating: /content/carside/train/labels/Auto681-D_NQ_NP_865756-MLA70215483386_062023-O-webp_jpg.rf.16624bdf6a5a4029a319010cf37b6f36.txt  
  inflating: /content/carside/train/labels/Auto687-D_NQ_NP_644559-MLA51718226871_092022-O-webp_

In [ ]:
!rm "/content/CarSide.v13i.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['carRear', 'carSide', 'car']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: carRear, carSide, car")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída baseada na classe"""
    class_to_folder = {
        0: 'carRear',
        1: 'car',
        2: 'carSide',
        3: 'carSide'
    }
    return class_to_folder.get(class_id, None)

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train ou valid)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Verifica se atende ao tamanho mínimo (56x56)
            if crop_width < 56 or crop_height < 56:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Determina a pasta de saída
            output_folder = get_output_folder(class_id)
            if output_folder is None:
                print(f"Aviso: Classe {class_id} não reconhecida")
                continue

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos, {skipped_crops} recortes ignorados (< 56x56)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens com Bounding Boxes Orientados (OBB)")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa as imagens de treino
    process_images('/content/carside/train/images', '/content/carside/train/labels', 'train')

    # Processa as imagens de validação
    process_images('/content/carside/valid/images', '/content/carside/valid/labels', 'valid')

    # Processa as imagens de teste
    process_images('/content/carside/test/images', '/content/carside/test/labels', 'test')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    for folder in ['carRear', 'carSide', 'car']:
        folder_path = base_output / folder
        if folder_path.exists():
            count = len(list(folder_path.glob('*.jpg')))
            print(f"Pasta '{folder}': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens com Bounding Boxes Orientados (OBB)
Pastas de saída criadas em /content/saida: carRear, carSide, car

Processando 6406 imagens de train...
train: 7520 recortes salvos, 0 recortes ignorados (< 56x56)

Processando 1951 imagens de valid...
valid: 2226 recortes salvos, 0 recortes ignorados (< 56x56)

Processando 2 imagens de test...
test: 1 recortes salvos, 0 recortes ignorados (< 56x56)

Processamento concluído!
Pasta 'carRear': 923 imagens
Pasta 'carSide': 6745 imagens
Pasta 'car': 2079 imagens


In [ ]:
!rm -rf "/content/carside/"

## 🧪 Curadoria de Imagens para CarRear

Repositório https://universe.roboflow.com/chaoyang-university-of-technology-z2mgc/carrear

Arquivo CarRear.v3i.yolov8-obb.zip.
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/CarRear.v3i.yolov8-obb.zip' '/content/CarRear.v3i.yolov8-obb.zip'

In [ ]:
!unzip "/content/CarRear.v3i.yolov8-obb.zip"  -d /content/carrear/

A saída de streaming foi truncada nas últimas 5000 linhas.
 extracting: /content/carrear/train/images/car_36_1330_jpg.rf.a9b244ea0a72b1380839dd47f65cab9d.jpg  
 extracting: /content/carrear/train/images/car_36_1370_jpg.rf.ae1e95379ff28747f4e3ad8fe432039e.jpg  
 extracting: /content/carrear/train/images/car_36_1400_jpg.rf.5540bebfeda613dad6401a80f9cdf61e.jpg  
 extracting: /content/carrear/train/images/car_36_1410_jpg.rf.4b254fde144f0a2d514661eec5a3b05a.jpg  
 extracting: /content/carrear/train/images/car_36_1730_jpg.rf.357240f168ab532bc38b3ee51e52b0e2.jpg  
 extracting: /content/carrear/train/images/car_36_190_jpg.rf.6c8fd04d1601e229f332905cc7829c5a.jpg  
 extracting: /content/carrear/train/images/car_36_200_jpg.rf.d4f707f0d6705fb83f74fb4bf6a4450a.jpg  
 extracting: /content/carrear/train/images/car_36_220_jpg.rf.b27fe3161212de762f4dd279f3b57646.jpg  
 extracting: /content/carrear/train/images/car_36_2270_jpg.rf.1740c9e7a6573afdad73dc94e7842dc4.jpg  
 extracting: /content/carrear/train

In [ ]:
!rm "/content/CarRear.v3i.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['carRear']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: carRear, carSide, car")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída baseada na classe"""
    class_to_folder = {
        0: 'carRear'
    }
    return class_to_folder.get(class_id, None)

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train ou valid)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Verifica se atende ao tamanho mínimo (56x56)
            if crop_width < 56 or crop_height < 56:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Determina a pasta de saída
            output_folder = get_output_folder(class_id)
            if output_folder is None:
                print(f"Aviso: Classe {class_id} não reconhecida")
                continue

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos, {skipped_crops} recortes ignorados (< 56x56)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens com Bounding Boxes Orientados (OBB)")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa as imagens de treino
    process_images('/content/carrear/train/images', '/content/carrear/train/labels', 'train')

    # Processa as imagens de validação
    process_images('/content/carrear/valid/images', '/content/carrear/valid/labels', 'valid')

    # Processa as imagens de teste
    process_images('/content/carrear/test/images', '/content/carrear/test/labels', 'test')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    for folder in ['carRear']:
        folder_path = base_output / folder
        if folder_path.exists():
            count = len(list(folder_path.glob('*.jpg')))
            print(f"Pasta '{folder}': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens com Bounding Boxes Orientados (OBB)
Pastas de saída criadas em /content/saida: carRear, carSide, car

Processando 2472 imagens de train...
train: 1674 recortes salvos, 0 recortes ignorados (< 56x56)

Processando 705 imagens de valid...
valid: 491 recortes salvos, 0 recortes ignorados (< 56x56)

Processando 353 imagens de test...
test: 252 recortes salvos, 0 recortes ignorados (< 56x56)

Processamento concluído!
Pasta 'carRear': 3340 imagens


In [ ]:
!rm -rf "/content/carrear/"

## 🧪 Curadoria de Imagens para motorbike

https://universe.roboflow.com/nguyen-quoc-anh/motorbike-lrhdp

Arquivo motobike.v2i.yolov8-obb.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/motobike.v2i.yolov8-obb.zip' '/content/motobike.v2i.yolov8-obb.zip'

In [ ]:
!unzip "/content/motobike.v2i.yolov8-obb.zip"  -d /content/motorbike/

Archive:  /content/motobike.v2i.yolov8-obb.zip
  inflating: /content/motorbike/README.dataset.txt  
  inflating: /content/motorbike/README.roboflow.txt  
  inflating: /content/motorbike/data.yaml  
   creating: /content/motorbike/train/
   creating: /content/motorbike/train/images/
 extracting: /content/motorbike/train/images/1413738078-2649_jpg.rf.3e2210251a7b0f3c93f203a6eead0f1a.jpg  
 extracting: /content/motorbike/train/images/1413738078-2649_jpg.rf.783f24014598195cc243cd813086377f.jpg  
 extracting: /content/motorbike/train/images/1413738078-2649_jpg.rf.b4e8e4f96d98356686724b6eaa73b885.jpg  
 extracting: /content/motorbike/train/images/2017-Bajaj-Pulsar-150-Specifications_jpg.rf.14cc3deac5b92eb431959e08591ce09e.jpg  
 extracting: /content/motorbike/train/images/2017-Bajaj-Pulsar-150-Specifications_jpg.rf.6a53c071bf1305e358eafe912f9e1bde.jpg  
 extracting: /content/motorbike/train/images/2017-Bajaj-Pulsar-150-Specifications_jpg.rf.91aa044b0f560f9d0eeca21da0f64e96.jpg  
 extracting:

In [ ]:
!rm "/content/motobike.v2i.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['motorbike']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: motorbike")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída baseada na classe"""
    class_to_folder = {
        0: 'motorbike'  # bike -> motorbike
    }
    return class_to_folder.get(class_id, None)

def get_min_size(class_id):
    """Retorna o tamanho mínimo baseado na classe"""
    min_sizes = {
        0: 72  # motorbike >= 72px
    }
    return min_sizes.get(class_id, 72)

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0
    skipped_by_class = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Filtra apenas classe 0 (bike)
            if class_id != 0:
                skipped_by_class += 1
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Obtém tamanho mínimo para a classe
            min_size = get_min_size(class_id)

            # Verifica se atende ao tamanho mínimo
            if crop_width < min_size or crop_height < min_size:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Determina a pasta de saída
            output_folder = get_output_folder(class_id)
            if output_folder is None:
                print(f"Aviso: Classe {class_id} não reconhecida")
                continue

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos")
    print(f"  - {skipped_crops} recortes ignorados (abaixo de 72px)")
    print(f"  - {skipped_by_class} objetos ignorados (classes 1 e 2)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens - Motorbike e Person")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa apenas as imagens de treino
    process_images('/content/motorbike/train/images', '/content/motorbike/train/labels', 'train')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    for folder in ['motorbike', 'person']:
        folder_path = base_output / folder
        if folder_path.exists():
            count = len(list(folder_path.glob('*.jpg')))
            print(f"Pasta '{folder}': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens - Motorbike e Person
Pastas de saída criadas em /content/saida: motorbike

Processando 736 imagens de train...
train: 707 recortes salvos
  - 35 recortes ignorados (abaixo de 72px)
  - 36 objetos ignorados (classes 1 e 2)

Processamento concluído!
Pasta 'motorbike': 707 imagens


In [ ]:
!rm -rf "/content/motorbike/"

## 🧪 Curadoria de Imagens para bicycle

Repositório https://universe.roboflow.com/my-workplace-1x5io/bicycle-hbmem

Arquivo bicycle5904.v1i.yolov8-obb.zip.
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/bicycle5904.v1i.yolov8-obb.zip' '/content/bicycle5904.v1i.yolov8-obb.zip'

In [ ]:
!unzip "/content/bicycle5904.v1i.yolov8-obb.zip"  -d /content/bicycle/

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/bicycle/train/labels/c-297-_png_jpg.rf.f5da9394906f94d52994c9d9b539057d.txt  
  inflating: /content/bicycle/train/labels/c-299-_png_jpg.rf.9d66103f42a4dcb2b12f8d300076ecda.txt  
  inflating: /content/bicycle/train/labels/c-299-_png_jpg.rf.a8f783751a827da349b4ea10278d5adf.txt  
  inflating: /content/bicycle/train/labels/c-299-_png_jpg.rf.c050555212195f039e0ce1ddbc3d6186.txt  
  inflating: /content/bicycle/train/labels/c-299-_png_jpg.rf.ef79e93f0d4f2718d4e163a27231c5f0.txt  
  inflating: /content/bicycle/train/labels/c-30-_png_jpg.rf.09281c0b5bd421dfc6bff09fe95a0dc0.txt  
  inflating: /content/bicycle/train/labels/c-30-_png_jpg.rf.2196bed9fa0705baff780f2f6ad84af0.txt  
  inflating: /content/bicycle/train/labels/c-30-_png_jpg.rf.766b8a3e2f4b6ba426e7d611f9aba51f.txt  
  inflating: /content/bicycle/train/labels/c-30-_png_jpg.rf.a7dae2025f5160a78fa36578be390c42.txt  
  inflating: /content/bicycle/train/labels/c-

In [ ]:
!rm -rf "/content/bicycle5904.v1i.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['bicycle']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: bicycle")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída baseada na classe"""
    class_to_folder = {
        0: 'bicycle',
        1: 'bicycle'
    }
    return class_to_folder.get(class_id, None)

def get_min_size(class_id):
    """Retorna o tamanho mínimo baseado na classe"""
    min_sizes = {
        0: 80,
        1: 80
    }
    return min_sizes.get(class_id, 80)

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0
    skipped_by_class = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Filtra apenas classes 0 e 1
            if class_id not in [0, 1]:
                skipped_by_class += 1
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Obtém tamanho mínimo para a classe
            min_size = get_min_size(class_id)

            # Verifica se atende ao tamanho mínimo
            if crop_width < min_size or crop_height < min_size:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Determina a pasta de saída
            output_folder = get_output_folder(class_id)
            if output_folder is None:
                print(f"Aviso: Classe {class_id} não reconhecida")
                continue

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos")
    print(f"  - {skipped_crops} recortes ignorados (abaixo de 80px)")
    print(f"  - {skipped_by_class} objetos ignorados (outras classes)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens - Bicycle")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa apenas as imagens de treino
    process_images('/content/bicycle/train/images', '/content/bicycle/train/labels', 'train')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    folder_path = base_output / 'bicycle'
    if folder_path.exists():
        count = len(list(folder_path.glob('*.jpg')))
        print(f"Pasta 'bicycle': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens - Bicycle
Pastas de saída criadas em /content/saida: bicycle

Processando 5904 imagens de train...
train: 7973 recortes salvos
  - 644 recortes ignorados (abaixo de 80px)
  - 0 objetos ignorados (outras classes)

Processamento concluído!
Pasta 'bicycle': 7973 imagens


In [ ]:
!rm -rf "/content/bicycle/"

## 🧪 Curadoria de Imagens para car

https://universe.roboflow.com/cardd-ziqmg/carfront

Arquivo carfront.v3i.yolov8.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/carfront.v3i.yolov8.zip' '/content/carfront.v3i.yolov8.zip'

In [ ]:
!unzip "/content/carfront.v3i.yolov8.zip"  -d /content/carFront/

Archive:  /content/carfront.v3i.yolov8.zip
  inflating: /content/carFront/README.dataset.txt  
  inflating: /content/carFront/README.roboflow.txt  
  inflating: /content/carFront/data.yaml  
   creating: /content/carFront/train/
   creating: /content/carFront/train/images/
 extracting: /content/carFront/train/images/Screenshot-148-_png.rf.7a2add4c683000a325305fea77e178b4.jpg  
 extracting: /content/carFront/train/images/Screenshot-164-_png.rf.6bfa829829f0baea0b23ced08f817895.jpg  
 extracting: /content/carFront/train/images/Screenshot-165-_png.rf.ee8386f65ab414e6f7a4d16b9cd00dff.jpg  
 extracting: /content/carFront/train/images/Screenshot-166-_png.rf.1ef3a762ac1c42bb484a70c480d4058c.jpg  
 extracting: /content/carFront/train/images/Screenshot-168-_png.rf.575f63eeebe9ca60814cf4482729cc0b.jpg  
 extracting: /content/carFront/train/images/Screenshot-169-_png.rf.4a64ba154c00f71bf4f09b6caf35cf56.jpg  
 extracting: /content/carFront/train/images/Screenshot-170-_png.rf.5472b8b589a3d5932e44a71

In [ ]:
!rm "/content/carfront.v3i.yolov8.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['car']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: car")

def polygon_to_bbox(coords, img_width, img_height):
    """
    Converte um polígono (múltiplos pontos x,y) para bounding box alinhado aos eixos

    Args:
        coords: lista com coordenadas [x1, y1, x2, y2, ...] normalizadas (0-1)
        img_width: largura da imagem em pixels
        img_height: altura da imagem em pixels

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Separa coordenadas x e y
    x_coords = []
    y_coords = []

    for i in range(0, len(coords), 2):
        x_coords.append(coords[i] * img_width)
        y_coords.append(coords[i + 1] * img_height)

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída (sempre 'car' independente da classe)"""
    return 'car'

def get_min_size(class_id):
    """Retorna o tamanho mínimo (56px para todas as classes)"""
    return 56

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos polígonos

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train/valid)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Deve ter pelo menos 3 valores (1 classe + pelo menos 1 par x,y)
            if len(parts) < 3:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])

                # Filtra apenas a classe 3
                if class_id != 3:
                    continue

                # Todos os valores restantes são coordenadas do polígono
                coords = list(map(float, parts[1:]))

                # Verifica se temos um número par de coordenadas (pares x,y)
                if len(coords) % 2 != 0:
                    print(f"Aviso: Número ímpar de coordenadas em {label_file}: {line.strip()}")
                    continue

            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Converte polígono para bounding box
            x_min, y_min, x_max, y_max = polygon_to_bbox(coords, img_width, img_height)

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Obtém tamanho mínimo para a classe
            min_size = get_min_size(class_id)

            # Verifica se atende ao tamanho mínimo
            if crop_width < min_size or crop_height < min_size:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Determina a pasta de saída
            output_folder = get_output_folder(class_id)

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos")
    print(f"  - {skipped_crops} recortes ignorados (abaixo de 56px)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens - Car (Formato Polígono)")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa as imagens de treino
    process_images('/content/carFront/train/images', '/content/carFront/train/labels', 'train')

    # Processa as imagens de validação
    process_images('/content/carFront/valid/images', '/content/carFront/valid/labels', 'valid')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    folder_path = base_output / 'car'
    if folder_path.exists():
        count = len(list(folder_path.glob('*.jpg')))
        print(f"Pasta 'car': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens - Car (Formato Polígono)
Pastas de saída criadas em /content/saida: car

Processando 108 imagens de train...
train: 107 recortes salvos
  - 0 recortes ignorados (abaixo de 56px)

Processando 27 imagens de valid...
valid: 27 recortes salvos
  - 0 recortes ignorados (abaixo de 56px)

Processamento concluído!
Pasta 'car': 2213 imagens


In [ ]:
!rm -rf "/content/carFront/"

## 🧪 Curadoria de Imagens para person

https://universe.roboflow.com/taisei/person-eccaa

person.v2i.yolov8-obb.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/person.v2i.yolov8-obb.zip' '/content/person.v2i.yolov8-obb.zip'

In [ ]:
!unzip "/content/person.v2i.yolov8-obb.zip"  -d /content/person/

Archive:  /content/person.v2i.yolov8-obb.zip
  inflating: /content/person/README.dataset.txt  
  inflating: /content/person/README.roboflow.txt  
  inflating: /content/person/data.yaml  
   creating: /content/person/test/
   creating: /content/person/test/images/
 extracting: /content/person/test/images/-48-_jpg.rf.2d36403275a24103be8da15177c12969.jpg  
 extracting: /content/person/test/images/-53-_jpg.rf.085e030c7c55bf8715c896ec0c676161.jpg  
 extracting: /content/person/test/images/-56-_jpg.rf.6587e24b7689d326d32c5e70911f0c9b.jpg  
 extracting: /content/person/test/images/20170314-01010108000002-01L_jpg.rf.696e6610800282a86335fea69fa71f43.jpg  
 extracting: /content/person/test/images/230903ae063093b6ceb2154e9917fc1e_webp.rf.39d362cd2508dd887660e5929587dde8.jpg  
 extracting: /content/person/test/images/49813656-_jpg.rf.a265e33d609c4c1b469bcb0e27e85461.jpg  
 extracting: /content/person/test/images/49815303-_jpg.rf.12af24bb3475e297a62b162c8548c823.jpg  
 extracting: /content/person/t

In [ ]:
!rm "/content/person.v2i.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria a pasta de saída para os recortes"""
    output_path = Path('/content/saida/person')
    os.makedirs(output_path, exist_ok=True)
    print(f"Pasta de saída criada em /content/saida/person")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train, valid ou test)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Filtra apenas classes 0 e 1
            if class_id not in [0, 1]:
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Verifica se atende ao tamanho mínimo (48x48)
            if crop_width < 48 or crop_height < 48:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida/person') / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos, {skipped_crops} recortes ignorados (< 48x48 ou classe inválida)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens - Person Dataset (Classes 0 e 1)")
    print("="*60)

    # Cria a pasta de saída
    create_output_folders()

    # Processa as imagens de treino
    process_images('/content/person/train/images', '/content/person/train/labels', 'train')

    # Processa as imagens de validação
    process_images('/content/person/valid/images', '/content/person/valid/labels', 'valid')

    # Processa as imagens de teste
    process_images('/content/person/test/images', '/content/person/test/labels', 'test')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    output_path = Path('/content/saida/person')
    if output_path.exists():
        count = len(list(output_path.glob('*.jpg')))
        print(f"Total de imagens salvas em 'person': {count}")

if __name__ == "__main__":
    main()

Recorte de Imagens - Person Dataset (Classes 0 e 1)
Pasta de saída criada em /content/saida/person

Processando 110 imagens de train...
train: 155 recortes salvos, 7 recortes ignorados (< 48x48 ou classe inválida)

Processando 32 imagens de valid...
valid: 50 recortes salvos, 21 recortes ignorados (< 48x48 ou classe inválida)

Processando 15 imagens de test...
test: 26 recortes salvos, 0 recortes ignorados (< 48x48 ou classe inválida)

Processamento concluído!
Total de imagens salvas em 'person': 231


In [ ]:
!rm -rf "/content/person/"

## 🧪 Curadoria de Imagens para motorbike, bicycle, car, carSide e carRear

https://www.kaggle.com/datasets/raghavdharwal/vehicle-orientation-dataset-part-1

kaggle_car.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/kaggle_car.zip' '/content/kaggle_car.zip'

In [ ]:
!unzip "/content/kaggle_car.zip"  -d /content/allcar/

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/allcar/vehicle-orientation-1/Y5RKFYJB44D74PONW64U.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5RKFYJB44D74PONW64U.txt  
  inflating: /content/allcar/vehicle-orientation-1/Y5RLKVQQXA6D226VA0ZC.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5RLKVQQXA6D226VA0ZC.txt  
  inflating: /content/allcar/vehicle-orientation-1/Y5RUVBYQBYYOFB5BUWDG.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5RUVBYQBYYOFB5BUWDG.txt  
  inflating: /content/allcar/vehicle-orientation-1/Y5S15S9PCZEV322IZ256.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5S15S9PCZEV322IZ256.txt  
  inflating: /content/allcar/vehicle-orientation-1/Y5SIZADQTMVRT1NKLFTE.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5SIZADQTMVRT1NKLFTE.txt  
  inflating: /content/allcar/vehicle-orientation-1/Y5SV68LJC8120G1ZWRF2.jpg  
  inflating: /content/allcar/vehicle-orientation-1/Y5SV68LJC8120G1ZWRF2.txt  
  inf

In [ ]:
!rm "/content/kaggle_car.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['carRear', 'carSide', 'car', 'motorbike', 'bicycle']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: {', '.join(folders)}")

def yolo_to_voc_bbox(x_center, y_center, width, height, img_width, img_height):
    """Convert YOLO normalized coordinates to Pascal VOC bbox format."""
    # Convert from normalized to pixel coordinates
    x_center_px = x_center * img_width
    y_center_px = y_center * img_height
    width_px = width * img_width
    height_px = height * img_height

    # Convert to VOC format (xmin, ymin, xmax, ymax)
    xmin = max(0, int(x_center_px - width_px / 2))
    ymin = max(0, int(y_center_px - height_px / 2))
    xmax = min(img_width, int(x_center_px + width_px / 2))
    ymax = min(img_height, int(y_center_px + height_px / 2))

    return xmin, ymin, xmax, ymax

def polygon_to_bbox(coords, img_width, img_height):
    """
    Converte coordenadas de polígono (normalizadas) para bounding box em pixels

    Args:
        coords: lista de coordenadas [x1, y1, x2, y2, ..., xn, yn] (normalizadas 0-1)
        img_width: largura da imagem em pixels
        img_height: altura da imagem em pixels

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Separa coordenadas x e y
    x_coords = []
    y_coords = []

    for i in range(0, len(coords), 2):
        x_coords.append(coords[i] * img_width)
        y_coords.append(coords[i+1] * img_height)

    # Encontra os valores mínimos e máximos
    x_min = max(0, int(min(x_coords)))
    x_max = min(img_width, int(max(x_coords)))
    y_min = max(0, int(min(y_coords)))
    y_max = min(img_height, int(max(y_coords)))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """
    Retorna a pasta de saída baseada na classe

    Args:
        class_id: ID da classe (0-14)

    Returns:
        Nome da pasta de saída ou None se a classe deve ser descartada
    """
    class_to_folder = {
        0: 'carRear',      # car_back
        1: 'carSide',      # car_side
        2: 'car',          # car_front
        9: 'motorbike',    # motorcycle_back
        10: 'motorbike',   # motorcycle_side
        11: 'motorbike',   # motorcycle_front
        12: 'bicycle',     # bicycle_back
        13: 'bicycle',     # bicycle_side
        14: 'bicycle'      # bicycle_front
    }
    return class_to_folder.get(class_id, None)

def get_min_size(class_id):
    """
    Retorna o tamanho mínimo baseado na classe

    Args:
        class_id: ID da classe (0-14)

    Returns:
        Tamanho mínimo em pixels
    """
    if class_id in [0, 1, 2]:  # Car
        return 56
    elif class_id in [9, 10, 11]:  # Motorbike
        return 72
    elif class_id in [12, 13, 14]:  # Bicycle
        return 80
    else:
        return 0

def process_images(base_path):
    """
    Processa as imagens e faz os recortes baseados nos polígonos

    Args:
        base_path: caminho para o diretório com imagens e labels
    """
    base_path = Path(base_path)

    if not base_path.exists():
        print(f"Erro: Pasta {base_path} não encontrada")
        return

    image_files = list(base_path.glob('*.jpg')) + list(base_path.glob('*.png')) + list(base_path.glob('*.jpeg'))

    # Contadores por categoria
    stats = {
        'carRear': {'total': 0, 'skipped': 0},
        'carSide': {'total': 0, 'skipped': 0},
        'car': {'total': 0, 'skipped': 0},
        'motorbike': {'total': 0, 'skipped': 0},
        'bicycle': {'total': 0, 'skipped': 0}
    }
    discarded_classes = 0

    print(f"\nProcessando {len(image_files)} imagens...")

    for img_idx, img_file in enumerate(image_files, 1):
        # Mostra progresso a cada 500 imagens
        if img_idx % 500 == 0 or img_idx == len(image_files):
            percent = (img_idx / len(image_files)) * 100
            print(f"Processando arquivo {img_idx}/{len(image_files)} ({percent:.1f}%)")
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = base_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Deve ter pelo menos 3 valores (1 classe + pelo menos 1 par x,y)
            if len(parts) < 3:
                continue

            try:
                class_id = int(parts[0])

                # Verifica se a classe deve ser processada
                output_folder = get_output_folder(class_id)
                if output_folder is None:
                    discarded_classes += 1
                    continue

                # Todos os valores restantes são coordenadas
                coords = list(map(float, parts[1:]))

                # Detecta o formato: 4 valores = YOLO bbox, mais valores = polígono
                if len(coords) == 4:
                    # Formato YOLO bbox (x_center, y_center, width, height)
                    x_min, y_min, x_max, y_max = yolo_to_voc_bbox(
                        coords[0], coords[1], coords[2], coords[3],
                        img_width, img_height
                    )
                else:
                    # Formato polígono
                    # Verifica se temos um número par de coordenadas (pares x,y)
                    if len(coords) % 2 != 0:
                        print(f"Aviso: Número ímpar de coordenadas em {label_file}: {line.strip()}")
                        continue

                    x_min, y_min, x_max, y_max = polygon_to_bbox(coords, img_width, img_height)

            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Obtém tamanho mínimo para a classe
            min_size = get_min_size(class_id)

            # Verifica se atende ao tamanho mínimo
            if crop_width < min_size or crop_height < min_size:
                stats[output_folder]['skipped'] += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Valida o recorte
            if cropped_img.size == 0:
                print(f"Aviso: Recorte vazio em {img_file}")
                continue

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            stats[output_folder]['total'] += 1

    # Exibe estatísticas
    print("\n" + "="*60)
    print("ESTATÍSTICAS DE PROCESSAMENTO")
    print("="*60)

    for folder, data in stats.items():
        print(f"\n{folder}:")
        print(f"  - Recortes salvos: {data['total']}")
        print(f"  - Recortes ignorados (tamanho): {data['skipped']}")

    print(f"\nClasses descartadas: {discarded_classes}")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens - Veículos Multi-classe")
    print("="*60)
    print("\nClasses processadas:")
    print("  0-car_back    → carRear   (min: 56px)")
    print("  1-car_side    → carSide   (min: 56px)")
    print("  2-car_front   → car       (min: 56px)")
    print("  9-motorcycle_back  → motorbike (min: 72px)")
    print("  10-motorcycle_side → motorbike (min: 72px)")
    print("  11-motorcycle_front → motorbike (min: 72px)")
    print("  12-bicycle_back → bicycle (min: 80px)")
    print("  13-bicycle_side → bicycle (min: 80px)")
    print("  14-bicycle_front → bicycle (min: 80px)")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa as imagens
    process_images('/content/allcar/vehicle-orientation-1')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas finais
    base_output = Path('/content/saida')
    for folder in ['carRear', 'carSide', 'car', 'motorbike', 'bicycle']:
        folder_path = base_output / folder
        if folder_path.exists():
            count = len(list(folder_path.glob('*.jpg')))
            print(f"Pasta '{folder}': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens - Veículos Multi-classe

Classes processadas:
  0-car_back    → carRear   (min: 56px)
  1-car_side    → carSide   (min: 56px)
  2-car_front   → car       (min: 56px)
  9-motorcycle_back  → motorbike (min: 72px)
  10-motorcycle_side → motorbike (min: 72px)
  11-motorcycle_front → motorbike (min: 72px)
  12-bicycle_back → bicycle (min: 80px)
  13-bicycle_side → bicycle (min: 80px)
  14-bicycle_front → bicycle (min: 80px)
Pastas de saída criadas em /content/saida: carRear, carSide, car, motorbike, bicycle

Processando 50000 imagens...
Processando arquivo 500/50000 (1.0%)
Processando arquivo 1000/50000 (2.0%)
Processando arquivo 1500/50000 (3.0%)
Processando arquivo 2000/50000 (4.0%)
Processando arquivo 2500/50000 (5.0%)
Processando arquivo 3000/50000 (6.0%)
Processando arquivo 3500/50000 (7.0%)
Processando arquivo 4000/50000 (8.0%)
Processando arquivo 4500/50000 (9.0%)
Processando arquivo 5000/50000 (10.0%)
Processando arquivo 5500/50000 (11.0%)
Processando arquivo 6000

In [ ]:
!rm -rf "/content/allcar/"

## 🧪 Curadoria de Imagens para Motorbike, bicycle e person

https://public.roboflow.com/object-detection/pascal-voc-

Pascal VOC 2012.v1-raw.yolov8-obb.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/Pascal VOC 2012.v1-raw.yolov8-obb.zip' '/content/Pascal VOC 2012.v1-raw.yolov8-obb.zip'

In [ ]:
!unzip "/content/Pascal VOC 2012.v1-raw.yolov8-obb.zip"  -d /content/pascalvoc/

A saída de streaming foi truncada nas últimas 5000 linhas.
 extracting: /content/pascalvoc/valid/images/2010_002589_jpg.rf.d001e826b4837fbc28b2dc8a581f9b5d.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002605_jpg.rf.e8ad08e55b510bdb1a579d3e5cc12012.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002615_jpg.rf.8af94178cd05b501f9f916893bb403a3.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002620_jpg.rf.ed51f3b42efde7fc71ac12facb5d0468.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002632_jpg.rf.cb6785492e2798fcde9773675eb625f6.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002652_jpg.rf.ccedca90bdce805e4565eb8e992d2276.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002661_jpg.rf.59c7bf0c5cbdcb9b717d15924c0d1daf.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002675_jpg.rf.11d8ec8598c7735b06d5b7a992cdfad7.jpg  
 extracting: /content/pascalvoc/valid/images/2010_002679_jpg.rf.5c1c3da13a16ade796b583420332731c.jpg  
 extracting: /

In [ ]:
!rm "/content/Pascal VOC 2012.v1-raw.yolov8-obb.zip"

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

def create_output_folders():
    """Cria as pastas de saída para os recortes"""
    base_output = Path('/content/saida')
    folders = ['bicycle', 'motorbike', 'person']
    for folder in folders:
        folder_path = base_output / folder
        os.makedirs(folder_path, exist_ok=True)
    print(f"Pastas de saída criadas em /content/saida: bicycle, motorbike, person")

def oriented_box_to_axis_aligned_bbox(points, img_width, img_height):
    """
    Converte uma caixa orientada (4 pontos) para um bounding box alinhado aos eixos

    Args:
        points: lista com 8 valores [x1, y1, x2, y2, x3, y3, x4, y4] normalizados
        img_width: largura da imagem
        img_height: altura da imagem

    Returns:
        x_min, y_min, x_max, y_max: coordenadas do bounding box em pixels
    """
    # Desnormaliza as coordenadas
    x_coords = [points[i] * img_width for i in range(0, 8, 2)]
    y_coords = [points[i] * img_height for i in range(1, 8, 2)]

    # Encontra os limites min/max
    x_min = int(min(x_coords))
    x_max = int(max(x_coords))
    y_min = int(min(y_coords))
    y_max = int(max(y_coords))

    return x_min, y_min, x_max, y_max

def get_output_folder(class_id):
    """Retorna a pasta de saída baseada na classe"""
    class_to_folder = {
        1: 'bicycle',
        13: 'motorbike',
        14: 'person'
    }
    return class_to_folder.get(class_id, None)

def get_min_size(class_id):
    """Retorna o tamanho mínimo baseado na classe"""
    min_sizes = {
        1: 80,      # bicycle
        13: 72,     # motorbike
        14: 48      # person
    }
    return min_sizes.get(class_id, 56)

def process_images(images_path, labels_path, dataset_name):
    """
    Processa as imagens e faz os recortes baseados nos bounding boxes orientados

    Args:
        images_path: caminho para a pasta de imagens
        labels_path: caminho para a pasta de labels
        dataset_name: nome do dataset (train ou valid)
    """
    images_path = Path(images_path)
    labels_path = Path(labels_path)

    if not images_path.exists():
        print(f"Aviso: Pasta {images_path} não encontrada")
        return

    if not labels_path.exists():
        print(f"Aviso: Pasta {labels_path} não encontrada")
        return

    image_files = list(images_path.glob('*.jpg')) + list(images_path.glob('*.png')) + list(images_path.glob('*.jpeg'))

    total_crops = 0
    skipped_crops = 0
    skipped_by_class = 0

    print(f"\nProcessando {len(image_files)} imagens de {dataset_name}...")

    for img_file in image_files:
        # Carrega a imagem
        img = cv2.imread(str(img_file))
        if img is None:
            print(f"Erro ao carregar {img_file}")
            continue

        img_height, img_width = img.shape[:2]

        # Busca o arquivo de label correspondente
        label_file = labels_path / f"{img_file.stem}.txt"

        if not label_file.exists():
            continue

        # Lê as anotações
        with open(label_file, 'r') as f:
            lines = f.readlines()

        for idx, line in enumerate(lines):
            parts = line.strip().split()

            # Verifica se tem exatamente 9 tokens (1 classe + 8 coordenadas)
            if len(parts) != 9:
                print(f"Aviso: Linha com formato incorreto em {label_file}: {line.strip()}")
                continue

            try:
                class_id = int(parts[0])
                # Os 8 valores restantes são as coordenadas dos 4 vértices
                coords = list(map(float, parts[1:9]))
            except ValueError:
                print(f"Erro ao converter valores em {label_file}: {line.strip()}")
                continue

            # Verifica se a classe está nas classes selecionadas
            output_folder = get_output_folder(class_id)
            if output_folder is None:
                skipped_by_class += 1
                continue

            # Converte a caixa orientada para bounding box alinhado aos eixos
            x_min, y_min, x_max, y_max = oriented_box_to_axis_aligned_bbox(
                coords, img_width, img_height
            )

            # Garante que as coordenadas estão dentro da imagem
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)

            # Calcula dimensões do recorte
            crop_width = x_max - x_min
            crop_height = y_max - y_min

            # Obtém o tamanho mínimo para a classe específica
            min_size = get_min_size(class_id)

            # Verifica se atende ao tamanho mínimo específico da classe
            if crop_width < min_size or crop_height < min_size:
                skipped_crops += 1
                continue

            # Faz o recorte
            cropped_img = img[y_min:y_max, x_min:x_max]

            # Salva a imagem recortada
            output_filename = f"{img_file.stem}_{idx}.jpg"
            output_path = Path('/content/saida') / output_folder / output_filename
            cv2.imwrite(str(output_path), cropped_img)

            total_crops += 1

    print(f"{dataset_name}: {total_crops} recortes salvos")
    print(f"  - {skipped_crops} recortes ignorados (tamanho < mínimo)")
    print(f"  - {skipped_by_class} objetos ignorados (classes não selecionadas)")

def main():
    """Função principal"""
    print("="*60)
    print("Recorte de Imagens Pascal VOC (OBB)")
    print("Classes: bicycle (>=80px), motorbike (>=72px), person (>=48px)")
    print("="*60)

    # Cria as pastas de saída
    create_output_folders()

    # Processa as imagens de treino
    process_images('/content/pascalvoc/train/images', '/content/pascalvoc/train/labels', 'train')

    # Processa as imagens de validação
    process_images('/content/pascavloc/valid/images', '/content/pascalvoc/valid/labels', 'valid')

    print("\n" + "="*60)
    print("Processamento concluído!")
    print("="*60)

    # Mostra estatísticas
    base_output = Path('/content/saida')
    for folder in ['bicycle', 'motorbike', 'person']:
        folder_path = base_output / folder
        if folder_path.exists():
            count = len(list(folder_path.glob('*.jpg')))
            print(f"Pasta '{folder}': {count} imagens")

if __name__ == "__main__":
    main()

Recorte de Imagens Pascal VOC (OBB)
Classes: bicycle (>=80px), motorbike (>=72px), person (>=48px)
Pastas de saída criadas em /content/saida: bicycle, motorbike, person

Processando 13690 imagens de train...
train: 10715 recortes salvos
  - 3788 recortes ignorados (tamanho < mínimo)
  - 16853 objetos ignorados (classes não selecionadas)
Aviso: Pasta /content/pascavloc/valid/images não encontrada

Processamento concluído!
Pasta 'bicycle': 11563 imagens
Pasta 'motorbike': 2113 imagens
Pasta 'person': 9959 imagens


In [ ]:
!rm -rf "/content/pascalvoc/"

## 🧪 Curadoria de Imagens para Motorbike

https://images.cv/download/motorbike/2301

motorbike.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/motorbike.zip' '/content/motorbike.zip'

In [ ]:
!unzip "/content/motorbike.zip"  -d /content/saida/

Archive:  /content/motorbike.zip
  inflating: /content/saida/motorbike/008S9HLEW13Z.jpg  
  inflating: /content/saida/motorbike/03DK7RK8WK5Z.jpg  
  inflating: /content/saida/motorbike/03TEKXLKG3G5.jpg  
  inflating: /content/saida/motorbike/04EZ6MKN6A5I.jpg  
  inflating: /content/saida/motorbike/04YO0ESR6MEY.jpg  
  inflating: /content/saida/motorbike/05KBQGCV0SE8.jpg  
  inflating: /content/saida/motorbike/08MRX8ZY45SQ.jpg  
  inflating: /content/saida/motorbike/08WWBENQUM7F.jpg  
  inflating: /content/saida/motorbike/09UUFCBUI8DW.jpg  
  inflating: /content/saida/motorbike/0BGEKPVS0CNG.jpg  
  inflating: /content/saida/motorbike/0CC6PR57N7WF.jpg  
  inflating: /content/saida/motorbike/0DBH82EUSZYB.jpg  
  inflating: /content/saida/motorbike/0DC3R0VJK38J.jpg  
  inflating: /content/saida/motorbike/0DCMGGYT0UNW.jpg  
  inflating: /content/saida/motorbike/0DEFWG6BZNUS.jpg  
  inflating: /content/saida/motorbike/0DUUF88AUUMM.jpg  
  inflating: /content/saida/motorbike/0ELLZSXN3OXW.jpg 

In [ ]:
!rm "/content/motorbike.zip"

## 🧪 Curadoria de Imagens para Person

https://images.cv/download/person/931

person.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/person.zip' '/content/person.zip'

In [ ]:
!unzip "/content/person.zip"  -d /content/saida/

Archive:  /content/person.zip
  inflating: /content/saida/person/01Z5BETRQ2ST.jpg  
  inflating: /content/saida/person/03CUTOYMMF1G.jpg  
  inflating: /content/saida/person/054W1UNWO68G.jpg  
  inflating: /content/saida/person/05AG88JYXLHW.jpg  
  inflating: /content/saida/person/05IOXEYRT3B9.jpg  
  inflating: /content/saida/person/06FNYARW9P6Q.jpg  
  inflating: /content/saida/person/07DST2QDOQFZ.jpg  
  inflating: /content/saida/person/07JJ6U0DT18S.jpg  
  inflating: /content/saida/person/0AL2BLUYLXB1.jpg  
  inflating: /content/saida/person/0BIQH4EJRQW9.jpg  
  inflating: /content/saida/person/0CA2ZCAUNJFI.jpg  
  inflating: /content/saida/person/0DD0XYSPW2LU.jpg  
  inflating: /content/saida/person/0H4PQLMOF0LO.jpg  
  inflating: /content/saida/person/0ID9NRFGRESV.jpg  
  inflating: /content/saida/person/0KD05GXE7DI6.jpg  
  inflating: /content/saida/person/0KXHOGQ17I1J.jpg  
  inflating: /content/saida/person/0L44B04ICVWR.jpg  
  inflating: /content/saida/person/0LEQG3WD0HUD.jpg 

In [ ]:
!rm "/content/person.zip/"

rm: cannot remove '/content/person.zip/': Not a directory


## 🧪 Curadoria de Imagens para Motorbike

https://www.kaggle.com/datasets/nqa112/vietnamese-bike-and-motorbike

archive.zip
---

In [ ]:
!cp '/content/drive/MyDrive/IC009/dataset/archive.zip' '/content/archive.zip'

In [ ]:
!unzip "/content/archive.zip"  -d /content/archive/

A saída de streaming foi truncada nas últimas 5000 linhas.
  inflating: /content/archive/motorbike/2017_08_04_04_02_37_BXW5VjhlTMB_20582471_349114885520621_6631977358167375872_n_1568719942018_19209.jpg  
  inflating: /content/archive/motorbike/2017_08_11_08_41_13_BXpayPADZvC_20688407_1348691611914953_4179012048115466240_n_1568719636400_14960.jpg  
  inflating: /content/archive/motorbike/2017_08_13_02_05_02_BXt3CMjFwrj_20759626_264752630696416_4927402995567558656_n_1568719636578_14962.jpg  
  inflating: /content/archive/motorbike/2017_08_16_00_38_34_BX1bhgdlFEn_20766451_457933204589480_6438631296439681024_n_1568719636772_14965.jpg  
  inflating: /content/archive/motorbike/2017_08_16_20_07_02_BX3hPsyDgEO_20766856_451478855236726_5117626439977402368_n_1568719636837_14966.jpg  
  inflating: /content/archive/motorbike/2017_08_18_13_52_09_BX7_7nQlq44_20902011_1191726327600417_2690989743306440704_n_1568719943740_19232.jpg  
  inflating: /content/archive/motorbike/2017_08_18_13_56_14_BX8AZhqlh

In [ ]:
!cp -rf '/content/archive/motorbike/' '/content/saida/'

In [ ]:
!rm -rf "/content/archive/"

In [ ]:
!rm "content/archive.zip"

rm: cannot remove 'content/archive.zip': No such file or directory


## 🧪 Verificando a quantidade de imagens em cada pasta
---

In [ ]:
import os
from pathlib import Path

def truncar_nomes_longos(pasta_principal, max_caracteres=200, tamanho_truncado=50):
    """
    Verifica e trunca nomes de arquivos que excedem o limite de caracteres.

    Args:
        pasta_principal: Caminho da pasta principal
        max_caracteres: Limite máximo de caracteres no nome do arquivo
        tamanho_truncado: Tamanho final do nome após truncamento
    """
    pasta = Path(pasta_principal)
    arquivos_renomeados = []

    # Percorre recursivamente todas as pastas e subpastas
    for item in pasta.rglob('*'):
        if item.is_file():
            nome_arquivo = item.name

            if len(nome_arquivo) > max_caracteres:
                # Separa nome e extensão
                extensao = item.suffix
                nome_sem_extensao = item.stem

                # Trunca o nome mantendo a extensão
                novo_nome = nome_sem_extensao[:tamanho_truncado - len(extensao)] + extensao
                novo_caminho = item.parent / novo_nome

                # Evita conflitos se o arquivo já existir
                contador = 1
                while novo_caminho.exists():
                    novo_nome = f"{nome_sem_extensao[:tamanho_truncado - len(extensao) - 3]}_{contador}{extensao}"
                    novo_caminho = item.parent / novo_nome
                    contador += 1

                # Renomeia o arquivo
                item.rename(novo_caminho)
                arquivos_renomeados.append({
                    'original': nome_arquivo,
                    'novo': novo_nome,
                    'pasta': str(item.parent.relative_to(pasta)),
                    'tamanho_original': len(nome_arquivo)
                })

    return arquivos_renomeados

def contar_arquivos_por_subpasta(pasta_principal):
    """
    Conta a quantidade de arquivos em cada subpasta.

    Args:
        pasta_principal: Caminho da pasta principal
    """
    pasta = Path(pasta_principal)

    # Verifica se a pasta existe
    if not pasta.exists():
        print(f"Erro: A pasta '{pasta_principal}' não existe!")
        return

    if not pasta.is_dir():
        print(f"Erro: '{pasta_principal}' não é uma pasta!")
        return

    # Primeiro, verifica e trunca nomes longos
    print("🔍 Verificando nomes de arquivos longos em todas as subpastas...")
    arquivos_renomeados = truncar_nomes_longos(pasta_principal)

    if arquivos_renomeados:
        print(f"\n{'='*80}")
        print(f"⚠️  {len(arquivos_renomeados)} arquivo(s) com nome longo foram renomeados:")
        print(f"{'='*80}\n")

        # Agrupa por subpasta
        por_subpasta = {}
        for item in arquivos_renomeados:
            subpasta = item['pasta']
            if subpasta not in por_subpasta:
                por_subpasta[subpasta] = []
            por_subpasta[subpasta].append(item)

        # Exibe agrupado por subpasta
        for subpasta, arquivos in sorted(por_subpasta.items()):
            print(f"📁 Subpasta: {subpasta}")
            print(f"   {len(arquivos)} arquivo(s) renomeado(s):\n")
            for item in arquivos:
                print(f"   • Original ({item['tamanho_original']} caracteres):")
                print(f"     {item['original'][:70]}{'...' if len(item['original']) > 70 else ''}")
                print(f"   • Novo nome:")
                print(f"     {item['novo']}\n")
            print()
    else:
        print("✅ Nenhum arquivo com nome longo encontrado.\n")

    # Dicionário para armazenar os resultados
    resultados = {}

    # Percorre todas as subpastas (nível 1 apenas)
    for subpasta in sorted(pasta.iterdir()):
        if subpasta.is_dir():
            # Conta apenas arquivos (não subpastas)
            num_arquivos = sum(1 for item in subpasta.iterdir() if item.is_file())
            resultados[subpasta.name] = num_arquivos

    # Exibe os resultados
    if not resultados:
        print(f"Nenhuma subpasta encontrada em '{pasta_principal}'")
        return

    print(f"\n{'='*60}")
    print(f"Quantidade de arquivos por subpasta em: {pasta_principal}")
    print(f"{'='*60}\n")

    total_arquivos = 0
    for subpasta, quantidade in resultados.items():
        print(f"{subpasta:40} → {quantidade:5} arquivo(s)")
        total_arquivos += quantidade

    print(f"\n{'-'*60}")
    print(f"{'TOTAL':40} → {total_arquivos:5} arquivo(s)")
    print(f"{'='*60}\n")

# Executa o programa
if __name__ == "__main__":
    pasta_principal = "/content/saida"
    contar_arquivos_por_subpasta(pasta_principal)

🔍 Verificando nomes de arquivos longos em todas as subpastas...

⚠️  8 arquivo(s) com nome longo foram renomeados:

📁 Subpasta: motorbike
   8 arquivo(s) renomeado(s):

   • Original (206 caracteres):
     81_300982_tem_xe_vespa_dep_tem_trum_vespa_lx_tem_che_vespa_sprint_dep_...
   • Novo nome:
     81_300982_tem_xe_vespa_dep_tem_trum_vespa_lx_t.jpg

   • Original (206 caracteres):
     cozsy3_I_asked_this_earlier_on_the_sub_and_got_a_different_picture__It...
   • Novo nome:
     cozsy3_I_asked_this_earlier_on_the_sub_and_got.jpg

   • Original (212 caracteres):
     86_232382_tem_xe_sh_150i_2010_moi_nhat_tem_trum_xe_sh_2010_tem_che_xe_...
   • Novo nome:
     86_232382_tem_xe_sh_150i_2010_moi_nhat_tem_tru.jpg

   • Original (221 caracteres):
     cu188h_My_baby_was_stolen_off_of_my_property_this_past_weekend__I_m_no...
   • Novo nome:
     cu188h_My_baby_was_stolen_off_of_my_property_t.jpg

   • Original (218 caracteres):
     79_300515_tem_xe_vario_2019_dep_tem_trum_vario_2018_moi_nh

## 🧪 Compacta a pasta /content/saida para uso posterior
---

In [ ]:
from google.colab import files

# 1. Compactar a pasta /content/saida no arquivo saida.zip
print("Iniciando a compactação da pasta...")
!zip -r saida.zip /content/saida
print("Compactação concluída: experiments.zip criado.")
print("-" * 30)
# 2. Baixar o arquivo saida.zip para o seu computador local
print("Iniciando o download do arquivo experiment_2.1.zip...")
!cp '/content/saida.zip' '/content/drive/MyDrive/IC009/dataset/processado/saida.zip'
print("Download iniciado. Verifique o progresso no seu navegador.")

A saída de streaming foi truncada nas últimas 5000 linhas.
  adding: content/saida/carRear/170FHWLJIG3TBQDYLSVN_5.jpg (deflated 0%)
  adding: content/saida/carRear/1BOQ79AWFNWQZJUSP014_1.jpg (deflated 5%)
  adding: content/saida/carRear/EGDLY3FRLMMD1AFNN013_3.jpg (deflated 6%)
  adding: content/saida/carRear/YQXB2SVJECWTV63FK98Z_2.jpg (deflated 6%)
  adding: content/saida/carRear/CUPXPBGC80L0PYH8H1TO_1.jpg (deflated 4%)
  adding: content/saida/carRear/Z20IQFLQX3GBAAM404XI_8.jpg (deflated 4%)
  adding: content/saida/carRear/EZWJ0BFVUPPIWUHEHE95_2.jpg (deflated 2%)
  adding: content/saida/carRear/YLDGJQLLK13WH6QWJ2AU_3.jpg (deflated 0%)
  adding: content/saida/carRear/9QKBHK3ZNO7IE8CB0DDE_3.jpg (deflated 3%)
  adding: content/saida/carRear/RB15GJ6ZNHP2M2BE83RN_3.jpg (deflated 0%)
  adding: content/saida/carRear/7Y8P6CYAU9PIU31NYTPF_7.jpg (deflated 1%)
  adding: content/saida/carRear/J4DS9V0HT0U62UZF59NC_5.jpg (deflated 0%)
  adding: content/saida/carRear/V0GP045RG6CNYX7MWZPE_0.jpg (defla

## 🧪 Seleciona 10.000 imagens aleatoriamente de cada pasta em /content/saida, reduz para 112x112 e grava em /content/kaggle
---

In [ ]:
import os
import random
from PIL import Image
from pathlib import Path
from collections import defaultdict

# Diretórios
dir_origem = '/content/saida'
dir_destino = '/content/kaggle'

# Criar diretório de destino se não existir
os.makedirs(dir_destino, exist_ok=True)

# 1. Coletar imagens organizadas por sub-pasta
print("Coletando imagens por sub-pasta...")
imagens_por_pasta = defaultdict(list)
extensoes_validas = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}

for root, dirs, files in os.walk(dir_origem):
    if root != dir_origem:  # Ignorar o diretório raiz
        sub_pasta = os.path.relpath(root, dir_origem)
        for file in files:
            if Path(file).suffix.lower() in extensoes_validas:
                caminho_completo = os.path.join(root, file)
                imagens_por_pasta[sub_pasta].append((caminho_completo, file))

print(f"Sub-pastas encontradas: {len(imagens_por_pasta)}")
for pasta, imgs in imagens_por_pasta.items():
    print(f"  {pasta}: {len(imgs)} imagens")

# 2. Processar cada sub-pasta separadamente para garantir 10.000 imagens
print("\nProcessando imagens para garantir 10.000 por sub-pasta...")
total_processadas = 0
total_erros = 0
total_erros_truncados = 0

for sub_pasta, lista_imagens in imagens_por_pasta.items():
    print(f"\nProcessando sub-pasta: {sub_pasta}")

    # Criar diretório de destino
    dir_destino_sub = os.path.join(dir_destino, sub_pasta)
    os.makedirs(dir_destino_sub, exist_ok=True)

    # Embaralhar lista completa
    random.shuffle(lista_imagens)

    processadas_pasta = 0
    erros_pasta = 0
    erros_truncados_pasta = 0
    idx = 0
    meta = 100000

    # Processar até atingir 10.000 imagens ou acabar a lista
    while processadas_pasta < meta and idx < len(lista_imagens):
        caminho_origem, nome_arquivo = lista_imagens[idx]
        idx += 1

        try:
            # Abrir imagem e carregar dados completamente
            img = Image.open(caminho_origem)
            img.load()  # Força o carregamento completo da imagem

            # Converter para RGB se necessário
            if img.mode == 'P':
                # Imagens em modo Palette - verificar se tem transparência
                if 'transparency' in img.info:
                    img = img.convert('RGBA')
                else:
                    img = img.convert('RGB')
            elif img.mode == 'RGBA':
                # Criar fundo branco para imagens com transparência
                background = Image.new('RGB', img.size, (255, 255, 255))
                background.paste(img, mask=img.split()[3])  # 3 é o canal alpha
                img = background
            elif img.mode not in ('RGB', 'L'):
                # Converter outros modos (CMYK, LA, etc.) para RGB
                img = img.convert('RGB')

            # Redimensionar imagem
            img_resized = img.resize((112, 112), Image.LANCZOS)

            # Mudar extensão para PNG
            nome_sem_extensao = Path(nome_arquivo).stem

            # Verificar o tamanho total do caminho completo que será criado
            caminho_teste = os.path.join(dir_destino_sub, f"{nome_sem_extensao}.png")

            # Se o caminho completo exceder 248 caracteres, truncar o nome
            if len(caminho_teste) > 248:
                # Calcular quanto precisa remover
                excesso = len(caminho_teste) - 248
                # Truncar o nome do arquivo (sem extensão)
                nome_sem_extensao = nome_sem_extensao[:-(excesso)]

            # Garantir que o nome tenha no máximo 240 caracteres
            if len(nome_sem_extensao) > 240:
                nome_sem_extensao = nome_sem_extensao[:240]

            nome_arquivo_png = f"{nome_sem_extensao}.png"

            # Salvar como PNG
            caminho_destino = os.path.join(dir_destino_sub, nome_arquivo_png)
            img_resized.save(caminho_destino, 'PNG')

            processadas_pasta += 1

            if processadas_pasta % 500 == 0:
                print(f"  Processadas: {processadas_pasta}/{meta}")

        except OSError as e:
            # Erros de arquivo truncado ou corrompido
            if "truncated" in str(e).lower():
                erros_truncados_pasta += 1
            else:
                erros_pasta += 1
        except Exception as e:
            erros_pasta += 1

    # Verificar se conseguiu processar 10.000
    if processadas_pasta < meta:
        print(f"  AVISO: Apenas {processadas_pasta} imagens válidas encontradas (meta: {meta})")
    else:
        print(f"  ✓ Meta atingida: {processadas_pasta} imagens processadas")

    print(f"  Erros (truncados): {erros_truncados_pasta}")
    print(f"  Erros (outros): {erros_pasta}")

    total_processadas += processadas_pasta
    total_erros += erros_pasta
    total_erros_truncados += erros_truncados_pasta

print(f"\n{'='*50}")
print(f"RESUMO GERAL:")
print(f"{'='*50}")
print(f"  Total processadas: {total_processadas}")
print(f"  Erros (arquivos truncados): {total_erros_truncados}")
print(f"  Erros (outros): {total_erros}")
print(f"  Total de erros: {total_erros + total_erros_truncados}")

# 3. Varrer sub-pastas e contar arquivos
print("\n" + "="*50)
print("Contagem de arquivos por sub-pasta em /content/kaggle:")
print("="*50)

total_geral = 0
contagem_pastas = defaultdict(int)

extensoes_validas_contagem = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}

for root, dirs, files in os.walk(dir_destino):
    if root != dir_destino:  # Ignorar o diretório raiz
        sub_pasta = os.path.relpath(root, dir_destino)
        qtd_arquivos = len([f for f in files if Path(f).suffix.lower() in extensoes_validas_contagem])
        if qtd_arquivos > 0:
            contagem_pastas[sub_pasta] = qtd_arquivos
            total_geral += qtd_arquivos

# Ordenar e exibir
for pasta in sorted(contagem_pastas.keys()):
    print(f"  {pasta}: {contagem_pastas[pasta]} arquivos")

print("="*50)
print(f"TOTAL: {total_geral} arquivos")
print("="*50)

Coletando imagens por sub-pasta...
Sub-pastas encontradas: 6
  person: 10889 imagens
  motorbike: 12464 imagens
  car: 41042 imagens
  carSide: 31310 imagens
  bicycle: 11563 imagens
  carRear: 69154 imagens

Processando imagens para garantir 10.000 por sub-pasta...

Processando sub-pasta: person
  Processadas: 500/100000
  Processadas: 1000/100000
  Processadas: 1500/100000
  Processadas: 2000/100000
  Processadas: 2500/100000
  Processadas: 3000/100000
  Processadas: 3500/100000
  Processadas: 4000/100000
  Processadas: 4500/100000
  Processadas: 5000/100000
  Processadas: 5500/100000
  Processadas: 6000/100000
  Processadas: 6500/100000
  Processadas: 7000/100000
  Processadas: 7500/100000
  Processadas: 8000/100000
  Processadas: 8500/100000
  Processadas: 9000/100000
  Processadas: 9500/100000
  Processadas: 10000/100000
  Processadas: 10500/100000
  AVISO: Apenas 10889 imagens válidas encontradas (meta: 100000)
  Erros (truncados): 0
  Erros (outros): 0

Processando sub-pasta: mo

## 🧪 Compacta a pasta /content/kaggle para uso na competição kaggle
---

In [ ]:
from google.colab import files

# 1. Compactar a pasta /content/kaggle no arquivo kaggle.zip
print("Iniciando a compactação da pasta...")
!zip -r kaggle.zip /content/kaggle
print("Compactação concluída: experiments.zip criado.")
print("-" * 30)
# 2. Baixar o arquivo kaggle.zip para o seu computador local
print("Iniciando o download do arquivo experiment_2.1.zip...")
!cp '/content/kaggle.zip' '/content/drive/MyDrive/IC009/dataset/processado/kaggle_all.zip'
print("Download iniciado. Verifique o progresso no seu navegador.")

In [ ]:
!rm -rf "/content/kaggle"